# COSC2753 Assignment 2 — Fashion Intelligence System
# Task 4 — Chained Test-Prediction Pipeline

**Prerequisite: run `COSC2753_A2_Preprocessing.ipynb` first.** This notebook loads the
artifacts that notebook writes to `../data/processed/` rather than repeating the cleaning,
splitting and encoding itself.

Loading rather than re-running matters for correctness, not just speed. Every notebook that
re-derives the split gets its own `StratifiedGroupKFold` draw, so a row could sit in train
here and validation there — and any score compared across notebooks would be measured
against a different validation set. Reading one saved split guarantees all four notebooks
mean the same thing by "validation".


## Part I — Load the preprocessed data

**MODIFY — this replaces the full preprocessing pipeline that used to be inlined here.**
The eighty-odd cells of Part I/II have been reduced to the three cells below, which read
what `COSC2753_A2_Preprocessing.ipynb` already computed and saved.

### What is loaded, and what is rebuilt

Loaded straight from disk:

| Artifact | File |
|---|---|
| Full cleaned train/validation split | `train_full.csv`, `val_full.csv` |
| Per-task usable subsets (target non-null, encoded) | `holdout_metadata/{target}_{train,val}.csv` |
| Fitted `LabelEncoder` per target | `label_encoders.pkl` |
| Image size, normalisation constants, thresholds, seed | `pipeline_config.json` |

Rebuilt in code, because they are not serialisable objects:

- **`train_transform` / `eval_transform`** — torchvision transforms hold a lambda-free but
  still unpicklable graph. They are reconstructed from the `normalization_mean` and
  `normalization_std` recorded in `pipeline_config.json`, so they are numerically identical
  to the ones used to compute those statistics, not merely similar.
- **`FashionImageDataset`, `get_weighted_sampler`** — ordinary class and function definitions.

### The one thing to watch

`id` is read back with `dtype=str`. Left to infer, pandas returns `int64`, and while
`f"{52003}.jpg"` happens to build a valid filename, an integer `id` breaks every
`set(...) & set(...)` leakage check against a string-keyed id set — silently, by finding no
overlap where overlap exists. The explicit dtype is what keeps those checks meaningful.


In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
from torch.utils.data import Dataset, WeightedRandomSampler
from torchvision import transforms as T
from sklearn.preprocessing import LabelEncoder

DATA_DIR = Path("../data/raw/FashionDataset")
OUT_DIR = Path("../data/processed")

TRAIN_CSV = DATA_DIR / "train" / "styles_train.csv"
IMAGES_TRAIN_DIR = DATA_DIR / "train" / "images_train"
TEST_PRED_CSV = DATA_DIR / "test" / "styles_prediction.csv"
IMAGES_TEST_DIR = DATA_DIR / "test" / "images_test"

REQUIRED = {
    "config":       OUT_DIR / "pipeline_config.json",
    "encoders":     OUT_DIR / "label_encoders.pkl",
    "train_full":   OUT_DIR / "train_full.csv",
    "val_full":     OUT_DIR / "val_full.csv",
    "holdout_dir":  OUT_DIR / "holdout_metadata",
}
missing = {k: v for k, v in REQUIRED.items() if not v.exists()}
for name, path in REQUIRED.items():
    print(f"[{'OK' if path.exists() else 'MISSING':>7}] {name:12s} {path}")
if missing:
    raise FileNotFoundError(
        "Run COSC2753_A2_Preprocessing.ipynb first -- it writes the files above. "
        f"Missing: {list(missing)}")

for p in [TRAIN_CSV, IMAGES_TRAIN_DIR, TEST_PRED_CSV, IMAGES_TEST_DIR]:
    print(f"[{'OK' if p.exists() else 'MISSING':>7}] {p}")

with open(REQUIRED["config"]) as f:
    config = json.load(f)

RANDOM_STATE = config["random_state"]
IMG_WIDTH = config["image"]["width"]
IMG_HEIGHT = config["image"]["height"]
mean = torch.tensor(config["image"]["normalization_mean"])
std = torch.tensor(config["image"]["normalization_std"])
ARTICLE_TYPE_MIN_COUNT = config["rare_class_thresholds"]["articleType"]
USAGE_MIN_COUNT = config["rare_class_thresholds"]["usage"]

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f"\nSeed {RANDOM_STATE} | images {IMG_HEIGHT}x{IMG_WIDTH} (HxW)")
print(f"Normalisation mean {[round(v, 4) for v in mean.tolist()]}, "
      f"std {[round(v, 4) for v in std.tolist()]}")
print(f"Rare-class thresholds: articleType>={ARTICLE_TYPE_MIN_COUNT}, usage>={USAGE_MIN_COUNT}")

In [ ]:
# dtype={'id': str} is load-bearing -- see the note above.
ID_DTYPE = {"id": str}

train_data = pd.read_csv(OUT_DIR / "train_full.csv", dtype=ID_DTYPE)
val_data = pd.read_csv(OUT_DIR / "val_full.csv", dtype=ID_DTYPE)

with open(OUT_DIR / "label_encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

holdout_dir = OUT_DIR / "holdout_metadata"
train_usable, val_usable = {}, {}
for t in encoders:
    train_usable[t] = pd.read_csv(holdout_dir / f"{t}_train.csv", dtype=ID_DTYPE)
    val_usable[t] = pd.read_csv(holdout_dir / f"{t}_val.csv", dtype=ID_DTYPE)

# `target_columns` maps a task name to the column actually predicted, which is NOT always
# the task name: rare-class grouping means articleType -> articleType_grouped and
# usage -> usage_grouped. Rebuilt by inspecting the saved files rather than hardcoded, so
# it cannot drift out of step with what preprocessing produced.
target_columns = {}
for t in encoders:
    candidates = [f"{t}_grouped", t]
    col = next((c for c in candidates
                if c in train_usable[t].columns and f"{c}_enc" in train_usable[t].columns), None)
    if col is None:
        raise KeyError(
            f"{t}: no encoded target column found. Looked for "
            f"{[c + '_enc' for c in candidates]} in {holdout_dir / f'{t}_train.csv'}. "
            "Re-run the preprocessing notebook to regenerate the holdout files.")
    target_columns[t] = col

# The main split partitions the cleaned dataset exactly, so `df` is recoverable by
# concatenation. Task 2's season split is a separate draw from the same rows (Section
# II.4.1) and is already reflected in the season_* holdout files, so it needs nothing here.
df = pd.concat([train_data, val_data], ignore_index=True)

print(f"train_full: {train_data.shape}   val_full: {val_data.shape}   combined: {df.shape}")
print("\nPer-task usable subsets:")
for t, col in target_columns.items():
    print(f"  {t:12s} -> column {col:20s} train={len(train_usable[t]):>6}  "
          f"val={len(val_usable[t]):>6}  classes={len(encoders[t].classes_):>3}")

# --- integrity checks: the saved split must still be a valid split -----------
assert set(train_data["id"]).isdisjoint(set(val_data["id"])), "id overlap between train and val"
assert set(train_data["dup_group"]).isdisjoint(set(val_data["dup_group"])), \
    "duplicate-image group spans the split"
assert df["id"].is_unique, "duplicate ids in the reloaded data"
for t, col in target_columns.items():
    enc_col = f"{col}_enc"
    n_cls = len(encoders[t].classes_)
    for name, frame in (("train", train_usable[t]), ("val", val_usable[t])):
        assert frame[col].notna().all(), f"{t}: null target in {name}"
        assert frame[enc_col].notna().all(), f"{t}: missing encoded label in {name}"
        assert frame[enc_col].between(0, n_cls - 1).all(), f"{t}: encoded label out of range in {name}"
        assert set(frame[col]).issubset(set(encoders[t].classes_)), \
            f"{t}: {name} contains a class the encoder never saw"
        # The encoded column must actually agree with the encoder, not merely be in range --
        # a stale CSV paired with a newer pickle would pass every check above.
        assert (encoders[t].transform(frame[col]) == frame[enc_col].to_numpy()).all(), \
            f"{t}: {enc_col} disagrees with label_encoders.pkl -- re-run preprocessing"
print("\nIntegrity checks passed: split is disjoint, ids unique, labels in range.")

In [ ]:
def to_rgb(img):
    """Converts the 249 grayscale ('L' mode) files to 3-channel. A module-level function,
    not a lambda, so the transform stays picklable for DataLoader(num_workers>0)."""
    return img.convert("RGB")


# Evaluation pipeline: no augmentation, so it stays a faithful evaluation signal.
eval_transform = T.Compose([
    to_rgb,
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.ToTensor(),
    T.Normalize(mean=mean.tolist(), std=std.tolist()),
])

# Training pipeline: conservative augmentation only. These are catalog product photos, so a
# vertical flip or a large rotation would produce an item that never occurs in the data.
# fill=255 because the backgrounds are white -- the default fill=0 would paste black wedges
# into every rotated corner and teach the model an artefact absent at evaluation time.
train_transform = T.Compose([
    to_rgb,
    T.Resize((IMG_HEIGHT, IMG_WIDTH)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10, fill=255),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=mean.tolist(), std=std.tolist()),
])


class FashionImageDataset(Dataset):
    """(image, label) pairs. `target_col` names an already-encoded integer column."""

    def __init__(self, dataframe, images_dir, target_col, transform):
        self.data = dataframe.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.target_col = target_col
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        with Image.open(self.images_dir / f"{row['id']}.jpg") as im:
            img = self.transform(im)
        return img, int(row[self.target_col])


def get_weighted_sampler(data, target_col):
    """Inverse-frequency WeightedRandomSampler. Train split only -- never apply to
    validation, which must keep the real class distribution to stay a fair estimate."""
    counts = data[target_col].value_counts()
    weights = data[target_col].map(lambda c: 1.0 / counts[c]).to_numpy(dtype="float64")
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


# Class weights, computed for reference. Imbalance is handled by the sampler above, not by
# weighting the loss -- applying both would double-count the correction.
class_weights = {}
for t, col in target_columns.items():
    vc = train_usable[t][col].value_counts()
    n_samples, n_classes = len(train_usable[t]), len(encoders[t].classes_)
    w = n_samples / (n_classes * vc)
    class_weights[t] = torch.tensor([w[c] for c in encoders[t].classes_], dtype=torch.float32)

# Verify the reconstructed pipeline against a real image before anything depends on it.
_probe_id = train_data["id"].iloc[0]
_probe = eval_transform(Image.open(IMAGES_TRAIN_DIR / f"{_probe_id}.jpg"))
assert _probe.shape == (3, IMG_HEIGHT, IMG_WIDTH), f"unexpected tensor shape {tuple(_probe.shape)}"
print(f"Transform check: image {_probe_id} -> tensor {tuple(_probe.shape)}, "
      f"range [{_probe.min():.2f}, {_probe.max():.2f}]")
print("Ready: train_data, val_data, df, train_usable, val_usable, encoders,")
print("       train_transform, eval_transform, FashionImageDataset, get_weighted_sampler")

## Part II — The Chained Pipeline

### The dependency problem, and how the chain avoids it

Each task's multi-input model needs metadata that the test set doesn't have:

| Model | Metadata it requires |
|---|---|
| Task 1 (`articleType`) | `gender`, `baseColour`, `season`, `usage`, `year` |
| Task 2 (`season`) | `gender`, `masterCategory`, `subCategory`, `articleType`, `baseColour`, `year`, `usage` |
| Task 3 (`gender`, `usage`) | `masterCategory`, `subCategory`, `articleType`, `baseColour`, `year`, `season` |

The union of what must be generated is eight columns. Note the circularity: Task 1 needs
`gender`, which is Task 3's target; Task 3 needs `articleType`, which is Task 1's target;
Task 2 needs both. **A multi-input model cannot supply another multi-input model's inputs** —
that's a cycle with no starting point.

The chain breaks it with two strictly ordered stages:

- **Stage 1 — attribute generation, image-only models only.** Every metadata column is
  produced from pixels alone. Nothing in this stage consumes metadata, so nothing is circular.
- **Stage 2 — final prediction, multi-input models.** These consume Stage 1's output. Nothing
  in Stage 2 feeds back into Stage 1.

Where each of the eight columns comes from:

| Column | Source | Notes |
|---|---|---|
| `articleType` | Task 1 best image-only CNN | from-scratch |
| `gender` | Task 3 image-only CNN | from-scratch |
| `usage` | Task 3 image-only CNN | from-scratch |
| `season` | Task 2 best image-only CNN | from-scratch |
| `baseColour` | **new CNN, trained in this notebook** | no task predicts it |
| `subCategory` | derived from predicted `articleType` | deterministic lookup, verified below |
| `masterCategory` | derived from predicted `subCategory` | deterministic lookup, verified below |
| `year` | mode of the training set | constant; no model |

### The honest caveat, stated up front

The multi-input models were trained on **true** metadata and will be served **predicted**
metadata. That is train/serve skew: a model that learned to trust a clean `gender` signal
will be fed a noisy one, and Task 1's own ablation (Step 11f) found `gender` was the single
largest contributor to its 0.750.

This pipeline always submits the chained multi-input model for every task — there is no
per-task fallback to image-only. That means the skew's cost has to be **measured and
reported honestly, not avoided**: Section 8 below scores the chain on the validation split
using *predicted* metadata, and that score — not the task notebooks' oracle scores — is what
should be quoted as expected test performance.

### 1. Configuration and artifact paths

Every model this notebook loads was trained and saved by Tasks 1–3. Run those three
notebooks first; this one trains only the `baseColour` model, which no task produces.

In [ ]:
import joblib
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, NUM_WORKERS = 128, 4
print("Device:", DEVICE)

TASK1_DIR = Path("../outputs/task1_models")
TASK2_DIR = Path("../outputs/task2_models")
TASK3_DIR = Path("../outputs/task3_models")
PIPELINE_DIR = Path("../outputs/pipeline")
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

# Fail early and specifically rather than deep inside a load call.
REQUIRED_ARTIFACTS = {
    "Task 1 image-only articleType": TASK1_DIR / "best_imageonly_articletype.pt",
    "Task 1 multi-input":            TASK1_DIR / "best_multiinput_task1.pt",
    "Task 1 metadata preprocessor":  TASK1_DIR / "metadata_preprocessor_task1.joblib",
    "Task 1 articleType encoder":    TASK1_DIR / "articletype_encoder_task1.joblib",
    "Task 2 multi-input":            TASK2_DIR / "best_multiinput_task2.pt",
    "Task 2 metadata preprocessor":  TASK2_DIR / "meta_preprocessor_task2.joblib",
    "Task 3 gender image-only":      TASK3_DIR / "gender_imageonly_smallcnn.pt",
    "Task 3 usage image-only":       TASK3_DIR / "usage_imageonly_smallcnn.pt",
    "Task 3 metadata one-hot":       TASK3_DIR / "ohe_metadata.joblib",
    "Task 3 gender encoder":         TASK3_DIR / "gender_encoder.joblib",
    "Task 3 usage encoder":          TASK3_DIR / "usage_encoder.joblib",
}

# Task 2 saves whichever image-only season model you trained; accept either.
TASK2_SEASON_CANDIDATES = [TASK2_DIR / "seresidual_cnn_image_only.pt",
                           TASK2_DIR / "improved_small_cnn_image_only.pt"]

missing = {k: v for k, v in REQUIRED_ARTIFACTS.items() if not v.exists()}
season_ckpt_path = next((p for p in TASK2_SEASON_CANDIDATES if p.exists()), None)

for name, path in REQUIRED_ARTIFACTS.items():
    print(f"[{'OK' if path.exists() else 'MISSING':>7}] {name:34s} {path}")
print(f"[{'OK' if season_ckpt_path else 'MISSING':>7}] {'Task 2 season image-only':34s} "
      f"{season_ckpt_path or TASK2_SEASON_CANDIDATES[0]}")

if missing or season_ckpt_path is None:
    raise FileNotFoundError(
        "Run Tasks 1-3 to completion first -- they produce the artifacts above. "
        f"Missing: {list(missing) + ([] if season_ckpt_path else ['Task 2 season image-only'])}")
print("\nAll required artifacts present.")

### 2. Architecture definitions

`torch.load` restores *weights*, not *structure*, so every architecture must be redeclared
here exactly as it was in the task notebooks. That duplication is a genuine fragility: edit
an encoder in a task notebook and this notebook will load stale weights into a mismatched
graph. Two defences:

- Every load below uses `strict=True` (PyTorch's default), so a shape or key mismatch raises
  rather than silently loading a partial state dict.
- Section 3 runs a forward pass on real images immediately after loading and checks the
  output shape against the expected class count.

If you keep iterating on architectures, move these classes into a shared `models.py` that all
four notebooks import — that removes the duplication entirely.

In [ ]:
import torch.nn.functional as F


# ── From Tasks 1 and 2: SE-residual building blocks ──────────────────────────
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        s = torch.sigmoid(self.fc2(F.silu(self.fc1(s))))
        return x * s[:, :, None, None]


class SEResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, drop=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.se = SqueezeExcite(out_ch)
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
        self.shortcut = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                         if (stride != 1 or in_ch != out_ch) else nn.Identity())

    def forward(self, x):
        out = F.silu(self.bn1(x))
        shortcut = self.shortcut(out if not isinstance(self.shortcut, nn.Identity) else x)
        out = self.conv1(out)
        out = self.conv2(F.silu(self.bn2(out)))
        out = self.se(self.drop(out))
        return out + shortcut


class SEResidualCNN(nn.Module):
    """Task 1's name for the encoder."""
    def __init__(self, out_dim=128, widths=(32, 64, 128, 256), drop=0.1):
        super().__init__()
        self.stem = nn.Conv2d(3, widths[0], 3, padding=1, bias=False)
        stages, in_ch = [], widths[0]
        for stage_idx, width in enumerate(widths):
            stride = 1 if stage_idx == 0 else 2
            stages.append(SEResidualBlock(in_ch, width, stride=stride, drop=drop))
            stages.append(SEResidualBlock(width, width, stride=1, drop=drop))
            in_ch = width
        self.stages = nn.Sequential(*stages)
        self.norm = nn.BatchNorm2d(in_ch)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_ch, out_dim))

    def forward(self, x):
        x = self.stages(self.stem(x))
        x = self.pool(F.silu(self.norm(x))).flatten(1)
        return self.proj(x)


SEResidualEncoder = SEResidualCNN   # Task 2's name for the identical architecture


# ── From Task 1 ──────────────────────────────────────────────────────────────
class BasicCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding='same'), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))


class VGGStyleCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding='same'), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding='same'), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding='same'), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding='same'), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, padding='same'), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding='same'), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2),
            nn.Conv2d(128, 256, 3, padding='same'), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding='same'), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Dropout2d(0.3),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = out_dim
        self.proj = nn.Linear(256, out_dim)

    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))


class ResNetStyleCNN(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import resnet18
        backbone = resnet18(weights=None)          # from scratch, as in Task 1
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.out_dim = out_dim
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class Classifier(nn.Module):
    """Task 1's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


class MetadataEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, out_dim), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class MultiInputNetT1(nn.Module):
    """Task 1's fusion model. Attribute names must match the saved state dict."""
    def __init__(self, image_encoder, meta_dim, n_classes, meta_embedding_dim=64):
        super().__init__()
        self.image_encoder = image_encoder
        self.meta_encoder = MetadataEncoder(meta_dim, out_dim=meta_embedding_dim)
        self.classifier = nn.Sequential(
            nn.Linear(image_encoder.out_dim + meta_embedding_dim, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, n_classes),
        )

    def forward(self, img, meta):
        return self.classifier(torch.cat([self.image_encoder(img), self.meta_encoder(meta)], dim=1))


# ── From Tasks 2 and 3 ───────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


class ImprovedSmallImageEncoder(nn.Module):
    def __init__(self, out_dim=128, dropout=0.2):
        super().__init__()
        self.out_dim = out_dim
        self.stage1 = ConvBlock(3, 32);   self.pool1 = nn.MaxPool2d(2)
        self.stage2 = ConvBlock(32, 64);  self.pool2 = nn.MaxPool2d(2)
        self.stage3 = ConvBlock(64, 128); self.pool3 = nn.MaxPool2d(2)
        self.stage4 = ConvBlock(128, 256)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, out_dim),
                                  nn.BatchNorm1d(out_dim), nn.SiLU())

    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        return self.proj(self.global_pool(self.stage4(x)).flatten(1))


class SmallImageEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.out_dim = out_dim
        self.proj = nn.Linear(128, out_dim)

    def forward(self, x):
        return self.proj(self.features(x).flatten(1))


class ResNet18Encoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import resnet18
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.out_dim = out_dim
        self.backbone = backbone
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class EfficientNetEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        from torchvision.models import efficientnet_b0
        backbone = efficientnet_b0(weights=None)
        self.out_dim = out_dim
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.proj = nn.Linear(1280, out_dim)

    def forward(self, x):
        return self.proj(self.backbone(x))


class MultiInputNetT23(nn.Module):
    """Tasks 2 and 3's fusion model. Note Task 2's version carries a learned
    `meta_scale` parameter; Task 3's does not. Both are handled at load time."""
    def __init__(self, metadata_dim, n_classes, image_encoder, meta_scale=False):
        super().__init__()
        self.image = image_encoder
        if meta_scale:
            self.meta_scale = nn.Parameter(torch.tensor(1.0))
        else:
            self.meta_scale = None
        self.meta = nn.Sequential(
            nn.BatchNorm1d(metadata_dim), nn.Linear(metadata_dim, 128),
            nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Sequential(
            nn.Linear(image_encoder.out_dim + 128, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, image, metadata):
        meta_feats = self.meta(metadata)
        if self.meta_scale is not None:
            meta_feats = meta_feats * self.meta_scale
        return self.head(torch.cat([self.image(image), meta_feats], dim=1))


class ImageOnlyClassifier(nn.Module):
    """Tasks 2 and 3's image-only wrapper."""
    def __init__(self, encoder, n_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(encoder.out_dim, n_classes)

    def forward(self, x):
        return self.head(self.encoder(x))


ENCODER_REGISTRY = {
    "SEResidualCNN": SEResidualCNN, "SEResidualEncoder": SEResidualEncoder,
    "BasicCNN": BasicCNN, "VGGStyleCNN": VGGStyleCNN, "ResNetStyleCNN": ResNetStyleCNN,
    "ImprovedSmallImageEncoder": ImprovedSmallImageEncoder,
    "SmallImageEncoder": SmallImageEncoder, "ResNet18Encoder": ResNet18Encoder,
    "EfficientNetEncoder": EfficientNetEncoder,
}
print(f"{len(ENCODER_REGISTRY)} encoder architectures registered.")

### 3. Datasets, loaders and the inference helper

One dataset class for inference on a bare list of ids — used identically for validation
images and test images, so the two paths cannot drift apart.

In [ ]:
class InferenceImageDataset(Dataset):
    """Images only, no labels. Returns (image_tensor, id) in the order given."""
    def __init__(self, ids, images_dir, transform):
        self.ids = list(ids)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img_id = self.ids[i]
        with Image.open(self.images_dir / f"{img_id}.jpg") as im:
            return self.transform(im), img_id


def make_inference_loader(ids, images_dir):
    return DataLoader(InferenceImageDataset(ids, images_dir, eval_transform),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


@torch.no_grad()
def predict_image_only(model, loader):
    """Returns predicted class INDICES in loader order. shuffle=False everywhere,
    so position i of the output corresponds to position i of the id list."""
    model.eval()
    preds = []
    for imgs, _ in tqdm(loader, desc="  image-only inference", leave=False):
        preds.append(model(imgs.to(DEVICE)).argmax(1).cpu())
    return torch.cat(preds).numpy()


@torch.no_grad()
def predict_multi_input(model, loader, meta_matrix):
    """Same, for a fusion model. `meta_matrix` must be row-aligned with the loader's
    id list -- the assertion below is the guard against a silent misalignment, which
    would produce plausible-looking but meaningless predictions."""
    model.eval()
    assert len(meta_matrix) == len(loader.dataset), \
        f"metadata rows ({len(meta_matrix)}) != images ({len(loader.dataset)})"
    preds, cursor = [], 0
    for imgs, _ in tqdm(loader, desc="  multi-input inference", leave=False):
        bs = imgs.size(0)
        meta = torch.as_tensor(meta_matrix[cursor:cursor + bs], dtype=torch.float32).to(DEVICE)
        preds.append(model(imgs.to(DEVICE), meta).argmax(1).cpu())
        cursor += bs
    assert cursor == len(meta_matrix), "consumed fewer metadata rows than expected"
    return torch.cat(preds).numpy()


def load_state(model, path_or_ckpt, label):
    """strict=True load with a clear failure message naming the culprit."""
    ckpt = torch.load(path_or_ckpt, map_location=DEVICE) if not isinstance(path_or_ckpt, dict) else path_or_ckpt
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as e:
        raise RuntimeError(
            f"{label}: saved weights do not match the architecture declared in Section 2. "
            f"This almost always means the encoder was edited in the task notebook after "
            f"the checkpoint was written -- retrain or sync the class definition.\n\n{e}")
    return model.to(DEVICE).eval()

### 4. Loading the Stage 1 attribute models

Each load is followed immediately by a forward pass on real validation images and a shape
check. A model that loads cleanly but produces the wrong number of logits would otherwise
only reveal itself much later, as a confusing indexing error.

In [ ]:
attribute_models = {}
val_probe_ids = val_data['id'].head(BATCH_SIZE).tolist()
probe_loader = make_inference_loader(val_probe_ids, IMAGES_TRAIN_DIR)


def register_attribute_model(name, model, encoder, n_expected):
    """Load-and-verify: run a real batch through and confirm the logit width."""
    with torch.no_grad():
        imgs, _ = next(iter(probe_loader))
        logits = model(imgs.to(DEVICE))
    assert logits.shape[1] == n_expected, \
        f"{name}: model outputs {logits.shape[1]} classes, encoder has {n_expected}"
    attribute_models[name] = (model, encoder)
    print(f"  [OK] {name:14s} {logits.shape[1]:3d} classes  ({type(model.encoder).__name__})")


print("Stage 1 attribute models:")

# --- articleType (Task 1) ---
ck = torch.load(TASK1_DIR / "best_imageonly_articletype.pt", map_location=DEVICE)
enc_at = joblib.load(TASK1_DIR / "articletype_encoder_task1.joblib")
m = Classifier(ENCODER_REGISTRY[ck["encoder_class"]](out_dim=ck["out_dim"]), ck["n_classes"])
register_attribute_model("articleType", load_state(m, ck, "Task 1 articleType"),
                         enc_at, len(enc_at.classes_))

# --- season (Task 2) ---
ck = torch.load(season_ckpt_path, map_location=DEVICE)
enc_se = encoders['season']
enc_cls = SEResidualEncoder if "seresidual" in season_ckpt_path.name else ImprovedSmallImageEncoder
m = ImageOnlyClassifier(enc_cls(out_dim=ck.get("out_dim", 128)), ck["n_classes"])
register_attribute_model("season", load_state(m, ck, "Task 2 season"), enc_se, len(enc_se.classes_))

# --- gender and usage (Task 3) ---
for attr, fname, enc_file in [("gender", "gender_imageonly_smallcnn.pt", "gender_encoder.joblib"),
                              ("usage", "usage_imageonly_smallcnn.pt", "usage_encoder.joblib")]:
    enc = joblib.load(TASK3_DIR / enc_file)
    m = ImageOnlyClassifier(ImprovedSmallImageEncoder(out_dim=128), len(enc.classes_))
    register_attribute_model(attr, load_state(m, TASK3_DIR / fname, f"Task 3 {attr}"),
                             enc, len(enc.classes_))

print(f"\n{len(attribute_models)} of 5 attribute models loaded "
      f"(baseColour is trained in Section 5).")

### 5. The missing model: `baseColour`

No task predicts `baseColour`, but Tasks 1, 2 and 3 all consume it — Task 1 flagged this as
outstanding work in its Step 14. It is trained here, using exactly the same machinery as
every other model: the shared split, the shared transforms, a `WeightedRandomSampler`, and a
from-scratch encoder.

`baseColour` has 46 categories in train with a long tail (the rarest has 2 rows), so the same
rare-class treatment used elsewhere applies: categories below a train-count threshold are
folded into `Other`. The threshold is set from **train counts only**, consistent with every
other threshold in this project.

In [ ]:
BASECOLOUR_MIN_COUNT = 50

vc_colour = train_data['baseColour'].value_counts()
print(f"baseColour categories in train: {len(vc_colour)}")
for thresh in [10, 25, 50, 100]:
    kept = (vc_colour >= thresh).sum()
    rows = vc_colour[vc_colour >= thresh].sum()
    print(f"  threshold={thresh:>4}: {kept:>2}/{len(vc_colour)} categories kept, "
          f"{rows} rows ({rows/len(train_data)*100:.1f}%)")

rare_colours = set(vc_colour[vc_colour < BASECOLOUR_MIN_COUNT].index)
for d in (train_data, val_data):
    d['baseColour_grouped'] = d['baseColour'].where(~d['baseColour'].isin(rare_colours), 'Other')

colour_encoder = LabelEncoder().fit(train_data['baseColour_grouped'])
N_COLOUR = len(colour_encoder.classes_)
train_data['baseColour_enc'] = colour_encoder.transform(train_data['baseColour_grouped'])

# Validation rows whose colour never appears in train can't be scored; none should exist
# after grouping, but check rather than assume.
val_known = val_data['baseColour_grouped'].isin(colour_encoder.classes_)
if (~val_known).sum():
    print(f"Dropping {(~val_known).sum()} val row(s) with an unseen baseColour")
    val_data = val_data[val_known].reset_index(drop=True)
val_data['baseColour_enc'] = colour_encoder.transform(val_data['baseColour_grouped'])

print(f"\nbaseColour classes after grouping: {N_COLOUR}")
print(train_data['baseColour_grouped'].value_counts())

In [ ]:
COLOUR_EPOCHS, COLOUR_PATIENCE = 30, 5

colour_train_ds = FashionImageDataset(train_data, IMAGES_TRAIN_DIR, 'baseColour_enc', train_transform)
colour_val_ds = FashionImageDataset(val_data, IMAGES_TRAIN_DIR, 'baseColour_enc', eval_transform)

colour_sampler = get_weighted_sampler(train_data, 'baseColour_enc')
colour_train_loader = DataLoader(colour_train_ds, batch_size=BATCH_SIZE, sampler=colour_sampler,
                                 num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))
colour_val_loader = DataLoader(colour_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, persistent_workers=(NUM_WORKERS > 0))


def fit_simple(model, loader_tr, loader_va, name, epochs, patience, lr=3e-4, weight_decay=1e-4):
    """Same recipe as the task notebooks: AdamW, ReduceLROnPlateau, early stopping on
    validation loss, best-checkpoint restore."""
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=2)
    best_loss, best_state, best_f1, best_epoch, bad = None, None, None, None, 0
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

    for epoch in range(epochs):
        for phase, loader in (('train', loader_tr), ('val', loader_va)):
            training = phase == 'train'
            model.train() if training else model.eval()
            total, n, preds, actual = 0.0, 0, [], []
            with torch.set_grad_enabled(training):
                for imgs, labels in tqdm(loader, desc=f'{name} {epoch+1}/{epochs} {phase}', leave=False):
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    logits = model(imgs)
                    loss = criterion(logits, labels)
                    if training:
                        optimiser.zero_grad(); loss.backward(); optimiser.step()
                    total += loss.item() * labels.size(0); n += labels.size(0)
                    preds.extend(logits.argmax(1).detach().cpu().numpy())
                    actual.extend(labels.cpu().numpy())
            epoch_loss, epoch_f1 = total / n, f1_score(actual, preds, average='macro', zero_division=0)
            history[f'{phase}_loss'].append(epoch_loss); history[f'{phase}_f1'].append(epoch_f1)

        val_loss, val_f1 = history['val_loss'][-1], history['val_f1'][-1]
        print(f'Epoch {epoch+1}/{epochs} - train_loss: {history["train_loss"][-1]:.4f} - '
              f'val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}')
        scheduler.step(val_loss)
        if best_loss is None or val_loss < best_loss:
            best_loss, best_f1, best_epoch, bad = val_loss, val_f1, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience:
                print(f'Early stopping at epoch {epoch+1} (best was {best_epoch})')
                break

    model.load_state_dict(best_state)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(name, fontweight='bold')
    axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='Train'); axes[1].plot(history['val_f1'], label='Val')
    axes[1].set_title('Macro-F1'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout(); plt.show()
    print(f'>>> {name}: best epoch {best_epoch}, val_loss={best_loss:.4f}, val_f1={best_f1:.4f}')
    return model, best_f1


import copy
torch.manual_seed(RANDOM_STATE)
colour_model = ImageOnlyClassifier(SEResidualCNN(out_dim=128), N_COLOUR).to(DEVICE)
colour_model, colour_f1 = fit_simple(colour_model, colour_train_loader, colour_val_loader,
                                     'baseColour CNN (from scratch)', COLOUR_EPOCHS, COLOUR_PATIENCE)

torch.save({'model_state_dict': colour_model.state_dict(), 'n_classes': N_COLOUR,
            'out_dim': 128, 'encoder_class': 'SEResidualCNN',
            'val_macro_f1': float(colour_f1), 'pretrained': False,
            'min_count': BASECOLOUR_MIN_COUNT},
           PIPELINE_DIR / 'basecolour_cnn.pt')
joblib.dump(colour_encoder, PIPELINE_DIR / 'basecolour_encoder.joblib')
attribute_models['baseColour'] = (colour_model, colour_encoder)
print(f"\nSaved baseColour model. {len(attribute_models)} attribute models ready.")

### 6. Derived and constant metadata

`subCategory` and `masterCategory` need no model. In this dataset each `articleType` belongs
to exactly one `subCategory`, and each `subCategory` to exactly one `masterCategory` — the
three columns form a fixed taxonomy, not three independent labels. So once `articleType` is
predicted, the other two follow by lookup.

That is an assumption about the data, so it is **verified rather than trusted**: the cell
below checks that each child maps to exactly one parent, and prints any violation instead of
silently taking the most common parent.

`year` gets the training mode. Task 1's Step 14 already named this as the plan, and it is the
right call — `year` is a catalogue timestamp with no visual signature, so a CNN would be
predicting noise. A constant is honest about carrying no information.

In [ ]:
# --- verify the taxonomy really is a function -------------------------------
at_to_sub = train_data.groupby('articleType')['subCategory'].nunique()
sub_to_master = train_data.groupby('subCategory')['masterCategory'].nunique()

ambiguous_at = at_to_sub[at_to_sub > 1]
ambiguous_sub = sub_to_master[sub_to_master > 1]

print(f"articleType values mapping to >1 subCategory:   {len(ambiguous_at)}")
print(f"subCategory values mapping to >1 masterCategory: {len(ambiguous_sub)}")
if len(ambiguous_at):
    print("\nAmbiguous articleType -> subCategory:"); print(ambiguous_at)
if len(ambiguous_sub):
    print("\nAmbiguous subCategory -> masterCategory:"); print(ambiguous_sub)

# Use the most common parent where ambiguity exists, but only after showing it above --
# the point is that the fallback is visible, not hidden.
ARTICLETYPE_TO_SUB = (train_data.groupby('articleType')['subCategory']
                      .agg(lambda s: s.mode().iloc[0]).to_dict())
SUB_TO_MASTER = (train_data.groupby('subCategory')['masterCategory']
                 .agg(lambda s: s.mode().iloc[0]).to_dict())

# The articleType model predicts articleType_grouped, whose rare values ARE subCategory
# names (Section II.3 folds rare types into their subCategory). Those map to themselves.
for sub in train_data['subCategory'].unique():
    ARTICLETYPE_TO_SUB.setdefault(sub, sub)

YEAR_MODE = float(pd.to_numeric(train_data['year'], errors='coerce').mode().iloc[0])

print(f"\nTaxonomy: {len(ARTICLETYPE_TO_SUB)} articleType -> subCategory, "
      f"{len(SUB_TO_MASTER)} subCategory -> masterCategory")
print(f"year constant (train mode): {YEAR_MODE:.0f}")

# Every class the articleType model can emit must have a subCategory mapping, or the
# pipeline would produce NaN for some rows.
unmapped = [c for c in attribute_models['articleType'][1].classes_ if c not in ARTICLETYPE_TO_SUB]
assert not unmapped, f"articleType classes with no subCategory mapping: {unmapped}"
print("All articleType classes resolve to a subCategory.")

### 7. Stage 1 — the metadata generator

One function, used for both validation and test. Everything downstream depends on this
returning a frame whose rows are in the same order as the ids it was given, so the id column
is returned alongside and checked by the caller.

In [ ]:
METADATA_COLUMNS = ['gender', 'masterCategory', 'subCategory', 'articleType',
                    'baseColour', 'season', 'usage', 'year']


def generate_metadata(ids, images_dir, verbose=True):
    """Stage 1: predict every metadata column from images alone.

    Returns a DataFrame indexed 0..n-1, row-aligned with `ids`. Uses ONLY image-only
    models, so nothing here depends on any multi-input model -- that is what keeps the
    chain acyclic.
    """
    ids = [str(i) for i in ids]
    loader = make_inference_loader(ids, images_dir)
    out = pd.DataFrame({'id': ids})

    for attr in ['articleType', 'gender', 'season', 'usage', 'baseColour']:
        model, encoder = attribute_models[attr]
        if verbose:
            print(f"  predicting {attr} ...")
        codes = predict_image_only(model, loader)
        out[attr] = encoder.inverse_transform(codes)

    # Derived, not predicted.
    out['subCategory'] = out['articleType'].map(ARTICLETYPE_TO_SUB)
    out['masterCategory'] = out['subCategory'].map(SUB_TO_MASTER)
    out['year'] = YEAR_MODE

    assert len(out) == len(ids) and list(out['id']) == ids, "row alignment broken"
    assert out[METADATA_COLUMNS].notna().all().all(), \
        f"NaN in generated metadata:\n{out[METADATA_COLUMNS].isna().sum()}"
    return out


# Smoke test on a handful of validation images before committing to the full run.
smoke_ids = val_data['id'].head(8).tolist()
smoke = generate_metadata(smoke_ids, IMAGES_TRAIN_DIR, verbose=False)
print("Smoke test -- generated metadata for 8 validation images:\n")
print(smoke[['id'] + METADATA_COLUMNS].to_string(index=False))
print("\nTrue values for the same 8 rows:\n")
print(val_data.head(8)[['id', 'gender', 'masterCategory', 'subCategory', 'articleType',
                        'baseColour', 'season', 'usage', 'year']].to_string(index=False))

### 8. Stage 1 attribute quality on validation

In [ ]:
# Stage 1 on the full validation split -- predicted metadata, never true values.
print("Generating predicted metadata for the validation split...")
val_ids = val_data['id'].tolist()
val_meta_pred = generate_metadata(val_ids, IMAGES_TRAIN_DIR)

# How good is each generated column? This is the leading indicator: if an attribute
# model is weak, the multi-input model that consumes it will be fed noise.
print("\nStage 1 attribute quality on validation (accuracy of each generated column):")
attr_quality = {}
truth = {'articleType': val_data['articleType_grouped'], 'gender': val_data['gender'],
         'season': val_data['season'], 'usage': val_data['usage_grouped'],
         'baseColour': val_data['baseColour_grouped']}
for attr, true_series in truth.items():
    mask = true_series.notna().to_numpy()
    acc = accuracy_score(true_series[mask], val_meta_pred.loc[mask, attr])
    f1m = f1_score(true_series[mask], val_meta_pred.loc[mask, attr],
                   average='macro', zero_division=0)
    attr_quality[attr] = {'accuracy': acc, 'macro_f1': f1m, 'n': int(mask.sum())}
    print(f"  {attr:12s} accuracy={acc:.3f}  macro-F1={f1m:.3f}  (n={mask.sum()})")

attr_quality_df = pd.DataFrame(attr_quality).T.round(4)
display(attr_quality_df)

### 9. Loading the Stage 2 multi-input models

In [ ]:
# ── Load the Stage 2 multi-input models ─────────────────────────────────────
multi_models = {}

# Task 1
info1 = json.load(open(TASK1_DIR / "best_multiinput_task1_info.json"))
enc_cls = ENCODER_REGISTRY[info1["image_encoder"]]
m1 = MultiInputNetT1(enc_cls(), meta_dim=info1["metadata_dim"], n_classes=info1["n_classes"])
multi_models['articleType'] = load_state(m1, TASK1_DIR / "best_multiinput_task1.pt", "Task 1 multi-input")
prep1 = joblib.load(TASK1_DIR / "metadata_preprocessor_task1.joblib")

# Task 2
ck2 = torch.load(TASK2_DIR / "best_multiinput_task2.pt", map_location=DEVICE)
m2 = MultiInputNetT23(ck2["metadata_dim"], ck2["n_classes"],
                      ENCODER_REGISTRY[ck2["image_encoder"]](out_dim=ck2["out_dim"]),
                      meta_scale=True)
multi_models['season'] = load_state(m2, ck2, "Task 2 multi-input")
prep2 = joblib.load(TASK2_DIR / "meta_preprocessor_task2.joblib")

# Task 3
ohe3 = joblib.load(TASK3_DIR / "ohe_metadata.joblib")
enc_g3 = joblib.load(TASK3_DIR / "gender_encoder.joblib")
enc_u3 = joblib.load(TASK3_DIR / "usage_encoder.joblib")
meta_dim3 = len(ohe3.get_feature_names_out())
for attr, fname, enc3 in [('gender', 'gender_multiinput_smallcnn.pt', enc_g3),
                          ('usage', 'usage_multiinput_smallcnn.pt', enc_u3)]:
    m3 = MultiInputNetT23(meta_dim3, len(enc3.classes_),
                          ImprovedSmallImageEncoder(out_dim=128), meta_scale=False)
    multi_models[attr] = load_state(m3, TASK3_DIR / fname, f"Task 3 {attr} multi-input")

print(f"Loaded {len(multi_models)} multi-input models: {list(multi_models)}")

TASK_ENCODER = {'articleType': attribute_models['articleType'][1], 'season': encoders['season'],
                'gender': enc_g3, 'usage': enc_u3}


# ── Build each task's metadata matrix with its OWN fitted preprocessor ───────
# Each task fit a different transformer on a different column set. Using the wrong one
# would produce a matrix of the wrong width (caught) or the right width with columns in
# the wrong order (NOT caught by any shape check) -- hence one builder per task.
T1_CAT, T1_NUM = ['gender', 'baseColour', 'season', 'usage'], ['year']
T2_COLS = ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'year', 'usage']
T2_CAT = [c for c in T2_COLS if c != 'year']
T3_COLS = ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'year', 'season']


def build_matrix_task1(meta_df):
    frame = meta_df[T1_CAT + T1_NUM].copy()
    for c in T1_CAT:
        frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce').astype(float)
    X = prep1.transform(frame)
    return X.toarray().astype('float32') if hasattr(X, 'toarray') else X.astype('float32')


def build_matrix_task2(meta_df):
    frame = meta_df[T2_COLS].copy()
    for c in T2_CAT:
        frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce')
    X = prep2.transform(frame)
    return X.toarray().astype('float32') if hasattr(X, 'toarray') else X.astype('float32')


def build_matrix_task3(meta_df):
    frame = meta_df[T3_COLS].copy()
    for c in T3_COLS:
        if c != 'year':
            frame[c] = frame[c].astype(object)
    frame['year'] = pd.to_numeric(frame['year'], errors='coerce')
    return ohe3.transform(frame).astype('float32')


MATRIX_BUILDERS = {'articleType': build_matrix_task1, 'season': build_matrix_task2,
                   'gender': build_matrix_task3, 'usage': build_matrix_task3}
EXPECTED_WIDTH = {'articleType': info1["metadata_dim"], 'season': ck2["metadata_dim"],
                  'gender': meta_dim3, 'usage': meta_dim3}

for task, builder in MATRIX_BUILDERS.items():
    X = builder(val_meta_pred)
    assert X.shape == (len(val_meta_pred), EXPECTED_WIDTH[task]), \
        (f"{task}: metadata matrix is {X.shape}, model expects "
         f"({len(val_meta_pred)}, {EXPECTED_WIDTH[task]}). The saved preprocessor and the "
         f"saved model came from different runs -- retrain or re-export both together.")
    print(f"  [OK] {task:12s} metadata matrix {X.shape}")

### 10. Chained validation quality

**This is the number to quote in the report as expected test performance.** The multi-input
model is scored on the validation split fed *predicted* metadata (Section 8's Stage 1 output)
— not the true metadata the task notebooks used, since the test set will never provide that.

In [ ]:
val_loader_pipe = make_inference_loader(val_ids, IMAGES_TRAIN_DIR)

TASK_TRUTH = {'articleType': val_data['articleType_grouped'], 'season': val_data['season'],
              'gender': val_data['gender'], 'usage': val_data['usage_grouped']}

chain_quality_rows = []
for task in ['articleType', 'season', 'gender', 'usage']:
    truth_series = TASK_TRUTH[task]
    mask = truth_series.notna().to_numpy()
    y_true = TASK_ENCODER[task].transform(truth_series[mask])

    chained = predict_multi_input(multi_models[task], val_loader_pipe,
                                  MATRIX_BUILDERS[task](val_meta_pred))[mask]
    macro_f1 = f1_score(y_true, chained, average='macro', zero_division=0)
    chain_quality_rows.append({'task': task, 'chained_macro_f1': round(float(macro_f1), 4)})
    print(f"  {task:12s} chained macro-F1 = {macro_f1:.4f}")

chain_quality = pd.DataFrame(chain_quality_rows).set_index('task')
display(chain_quality)

print("\nFor context (not what gets submitted): each task notebook's own reported score used")
print("TRUE metadata. The gap between that number and chained_macro_f1 above is the cost of")
print("the train/serve skew -- worth naming in the report, not worth hiding.")

### 11. Structural checks on the test metadata

This section asks whether the pipeline *runs correctly* on the real test set — a different
question from Section 10's *is it any good*, and one that has to pass regardless.

Nine checks, each of which catches a failure that would otherwise produce a plausible-looking
but wrong submission file.

In [ ]:
test_df = pd.read_csv(TEST_PRED_CSV)
test_df['id'] = test_df['id'].astype(str).str.strip()
TEST_TEMPLATE_COLUMNS = list(test_df.columns)
print(f"Test template: {test_df.shape[0]} rows, columns {TEST_TEMPLATE_COLUMNS}")

checks = []


def check(name, passed, detail=""):
    checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))
    return passed


# 1. Every test id has an image file.
missing_imgs = [i for i in test_df['id'] if not (IMAGES_TEST_DIR / f"{i}.jpg").exists()]
check("every test id has an image file", not missing_imgs,
      f"{len(missing_imgs)} missing" if missing_imgs else f"{len(test_df)} images found")

# 2. Test ids are unique.
check("test ids are unique", test_df['id'].is_unique,
      f"{test_df['id'].duplicated().sum()} duplicates")

# 3. Test ids don't overlap the training set (they shouldn't -- separate folders).
overlap = set(test_df['id']) & set(df['id'].astype(str))
check("no test id appears in the training data", not overlap, f"{len(overlap)} overlapping")

# 4. Every test image opens and is the expected size.
bad_imgs, sizes = [], []
for img_id in test_df['id'].head(500):
    try:
        with Image.open(IMAGES_TEST_DIR / f"{img_id}.jpg") as im:
            sizes.append(im.size)
    except Exception as e:
        bad_imgs.append((img_id, str(e)))
check("test images open cleanly (500-image sample)", not bad_imgs,
      f"most common size {pd.Series(sizes).value_counts().index[0]}" if sizes else "")

# 5. A single batch survives the full transform.
probe = make_inference_loader(test_df['id'].head(BATCH_SIZE).tolist(), IMAGES_TEST_DIR)
probe_imgs, probe_ids = next(iter(probe))
check("test images pass through eval_transform",
      probe_imgs.shape[1:] == (3, IMG_HEIGHT, IMG_WIDTH),
      f"batch tensor {tuple(probe_imgs.shape)}")

# 6. Loader preserves id order (the assumption every alignment depends on).
check("DataLoader preserves id order",
      list(probe_ids) == test_df['id'].head(BATCH_SIZE).tolist())

In [ ]:
# ── Stage 1 on the real test set ─────────────────────────────────────────────
print("Generating metadata for the test set...")
test_meta = generate_metadata(test_df['id'].tolist(), IMAGES_TEST_DIR)

# 7. No missing values anywhere in the generated metadata.
check("generated test metadata has no NaN", test_meta[METADATA_COLUMNS].notna().all().all(),
      f"{int(test_meta[METADATA_COLUMNS].isna().sum().sum())} NaN cells")

# 8. Every generated category was seen during training -- an unseen value would be
#    silently dropped to all-zeros by handle_unknown='ignore' and quietly degrade results.
unseen_report = {}
for col in ['gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'usage']:
    train_vals = set(train_data[col].dropna().astype(str)) if col in train_data.columns else set()
    if col == 'articleType':
        train_vals = set(train_data['articleType_grouped'].dropna().astype(str))
    if col == 'usage':
        train_vals = set(train_data['usage_grouped'].dropna().astype(str))
    if col == 'baseColour':
        train_vals = set(train_data['baseColour_grouped'].dropna().astype(str))
    unseen = set(test_meta[col].astype(str)) - train_vals
    unseen_report[col] = unseen
check("no generated value is unseen in training",
      not any(unseen_report.values()),
      "; ".join(f"{k}: {v}" for k, v in unseen_report.items() if v) or "all values known")

# 9. Each task's metadata matrix has the exact width its model expects.
matrix_ok = True
for task, builder in MATRIX_BUILDERS.items():
    X = builder(test_meta)
    ok = X.shape == (len(test_meta), EXPECTED_WIDTH[task])
    matrix_ok &= ok
    print(f"      {task:12s} matrix {X.shape}, expected "
          f"({len(test_meta)}, {EXPECTED_WIDTH[task]})")
check("metadata matrices match the saved models' input widths", matrix_ok)

validation_report = pd.DataFrame(checks)
display(validation_report)

n_failed = (validation_report['result'] == 'FAIL').sum()
if n_failed:
    raise AssertionError(f"{n_failed} structural check(s) failed -- fix before submitting. "
                         "See the table above.")
print("\nAll structural checks passed.")

print("\nGenerated test metadata -- distribution of each column:")
for col in ['gender', 'season', 'usage', 'baseColour']:
    print(f"\n{col}:"); print(test_meta[col].value_counts().head(8))

### 12. Final predictions

Stage 2. For each of the four target columns, the chained multi-input model runs on the test
images using Section 7's generated metadata. No per-task decision, no image-only fallback —
this pipeline always submits the chain.

In [ ]:
test_loader = make_inference_loader(test_df['id'].tolist(), IMAGES_TEST_DIR)
submission = test_df[['id']].copy()

for task in ['articleType', 'season', 'gender', 'usage']:
    print(f"Predicting {task} (chained multi-input)...")
    codes = predict_multi_input(multi_models[task], test_loader, MATRIX_BUILDERS[task](test_meta))
    submission[task] = TASK_ENCODER[task].inverse_transform(codes)

submission.head()

### 13. Final submission checks

In [ ]:
# ── Final submission checks ──────────────────────────────────────────────────
submission = submission[TEST_TEMPLATE_COLUMNS]   # exact template column order

final_checks = []


def final_check(name, passed, detail=""):
    final_checks.append({'check': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" -- {detail}" if detail else ""))


final_check("column names and order match the template",
            list(submission.columns) == TEST_TEMPLATE_COLUMNS, str(list(submission.columns)))
final_check("row count matches the template",
            len(submission) == len(test_df), f"{len(submission)} vs {len(test_df)}")
final_check("id order matches the template",
            list(submission['id']) == list(test_df['id']))
final_check("no empty predictions",
            submission[['gender', 'articleType', 'season', 'usage']].notna().all().all(),
            f"{int(submission.isna().sum().sum())} NaN cells")
for col, enc in [('gender', enc_g3), ('season', encoders['season']),
                 ('usage', enc_u3), ('articleType', TASK_ENCODER['articleType'])]:
    bad = set(submission[col]) - set(enc.classes_)
    final_check(f"every predicted {col} is a valid class label", not bad, str(bad) if bad else "")

# A model that collapsed to one class would still pass every check above.
for col in ['gender', 'articleType', 'season', 'usage']:
    n_unique = submission[col].nunique()
    top_share = submission[col].value_counts(normalize=True).iloc[0]
    final_check(f"{col} predictions are not degenerate", n_unique > 1 and top_share < 0.95,
                f"{n_unique} distinct values, most common {top_share:.1%}")

display(pd.DataFrame(final_checks))
if (pd.DataFrame(final_checks)['result'] == 'FAIL').any():
    raise AssertionError("Submission checks failed -- do not submit this file.")

SUBMISSION_PATH = PIPELINE_DIR / 'styles_prediction_filled.csv'
submission.to_csv(SUBMISSION_PATH, index=False)
test_meta.to_csv(PIPELINE_DIR / 'test_generated_metadata.csv', index=False)
chain_quality.to_csv(PIPELINE_DIR / 'chain_validation_quality.csv')

# Reload check: what's on disk is what we think it is.
reloaded = pd.read_csv(SUBMISSION_PATH)
reloaded['id'] = reloaded['id'].astype(str)
assert list(reloaded.columns) == TEST_TEMPLATE_COLUMNS and len(reloaded) == len(test_df)
assert reloaded[['gender', 'articleType', 'season', 'usage']].notna().all().all()

print(f"\nWrote {len(submission)} predictions to {SUBMISSION_PATH}")
print(f"Wrote generated metadata to {PIPELINE_DIR / 'test_generated_metadata.csv'}")
print(f"Wrote validation quality report to {PIPELINE_DIR / 'chain_validation_quality.csv'}")
print("\nFirst rows of the submission:")
print(submission.head(10).to_string(index=False))

### 14. What to write in the report

**What this pipeline always does.** Every final prediction comes from the chained
multi-input model — Stage 1 predicts the metadata columns from images, Stage 2's multi-input
model consumes that predicted metadata. There's no per-task fallback to an image-only model,
so this choice needs to be justified once, for the whole pipeline, rather than re-litigated
per task.

**Two numbers to quote, not one.** Section 8's Stage 1 attribute quality (accuracy of each
generated column) tells the marker how much noise Stage 2 is fed. Section 10's chained
macro-F1 on validation is the actual expected test performance — quote this, not the multi-input
model's oracle score from the task notebooks. That oracle number was measured with true
metadata and will not be reproduced at test time.

**The known limitation, worth naming plainly.** The multi-input models were trained on clean,
true metadata and are served predicted metadata here — a train/serve skew. If Section 10's
chained score sits noticeably below the task notebook's reported (oracle) score, that gap is
the cost of the skew. Naming this in the report (rather than only quoting the higher number)
is exactly the kind of critical analysis the rubric is asking for.

**If you want to go further.** The principled fix for the skew is to retrain the multi-input
models on *cross-fitted predicted* metadata: split the training set into k folds, train the
attribute models on k−1 folds, predict the held-out fold, and assemble a training set whose
metadata carries the same kind of error the test set will see. That removes the skew instead
of merely measuring it. It costs k times the attribute-model training, so it's reasonable to
name it as future work rather than implement it here.